<a href="https://colab.research.google.com/github/letuanlocc/training_model/blob/main/notebook_colab_team.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PhoBERT NER — Colab cho team (mỗi người tự train)

`Runtime → Change runtime type → T4 GPU` trước khi chạy. Chạy cell theo thứ tự từ trên xuống.

In [2]:
#@title 1. Clone repo + cd vào project
!git clone https://github.com/letuanlocc/training_model.git /content/training_model
%cd /content/training_model/phobert_ner
!ls

Cloning into '/content/training_model'...
remote: Enumerating objects: 121, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 121 (delta 30), reused 120 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (121/121), 1.74 MiB | 18.98 MiB/s, done.
Resolving deltas: 100% (30/30), done.
/content/training_model/phobert_ner
CONTRIBUTING.md       notebook_colab_team.ipynb  train_crf.sh
data_build	      repo_changes.diff		 train_lstm_crf.sh
data_dir	      requirements_working.txt	 train_softmax.sh
notebook_colab.ipynb  SETUP_REPORT.md		 VPhoBertTagger


In [3]:
#@title 3. Cài môi trường (Java + Python packages + VnCoreNLP)
!apt-get -qq update && apt-get -qq install -y openjdk-17-jre-headless > /dev/null

!grep -v '^#' requirements_working.txt | grep -v -E '^(numpy|pandas|matplotlib|scikit-learn)' | sed '/^$/d' > /tmp/req_colab.txt
!pip -q install -r /tmp/req_colab.txt

import os
VN = os.path.abspath('vncorenlp'); B = 'https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master'
os.makedirs(f'{VN}/models/wordsegmenter', exist_ok=True)
for f in ['VnCoreNLP-1.2.jar', 'VnCoreNLP-1.1.1.jar', 'models/wordsegmenter/vi-vocab', 'models/wordsegmenter/wordsegmenter.rdr']:
    !wget -q -nc -O {VN}/{f} {B}/{f}
print('Xong.')

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have hugging

## 4. Tự tạo file dữ liệu của bạn

Mở panel **Files** bên trái Colab → `training_model/phobert_ner/data_build/` → chuột phải → **New file** →
đặt tên `handwritten_<ten>.txt` → viết câu, mỗi dòng 1 câu, đánh dấu thực thể bằng `[text|TAG]`
(TAG là `SKILL`/`TIME`/`LOC`). Xem ví dụ có sẵn trong `handwritten.txt` hoặc đọc `CONTRIBUTING.md`.

Lưu file xong mới chạy tiếp bước dưới.

## 5. Sinh dữ liệu + Train

Chọn 1 kiến trúc (softmax nhanh nhất, crf/lstm_crf thường chính xác hơn). ~15-30 phút trên GPU T4.

In [19]:
!ls -la data_build/handwritten*.txt


-rw-r--r-- 1 root root 20426 Nov  5 09:10 data_build/handwritten.txt


In [4]:
ARCH = "softmax"  #@param ["softmax", "crf", "lstm_crf"]

!python data_build/build_data.py
!python data_build/validate_data.py
!bash train_{ARCH}.sh 2>&1 | grep -v 'it/s]'

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
2026-11-05 09:11:56 INFO  WordSegmenter:24 - Loading Word Segmentation model
Câu viết tay gốc: 298 | sau tăng cường căn được: 1171 | lỗi căn: 42
Câu template căn được: 736 | lỗi căn: 12 | số pattern: 99
Bị loại do trùng lặp: 35
train: 1503 câu | handwritten=915 template=588 (39.1% template) | nhóm=316
dev: 185 câu | handwritten=111 template=74 (40.0% template) | nhóm=42
test: 184 câu | handwritten=113 template=71 (38.6% template) | nhóm=38
test_hard: 102 câu | lỗi căn: 4 | trùng nguyên văn với train: 0
TỔNG lỗi căn (alignment_failures.jsonl): 58
Nhóm nằm ở >1 tập (phải = 0): 0
tập       | số câu | số token | SKILL | TIME | LOC | không thực thể | không dấu | có ph

## 6. Lưu model lên Drive

In [5]:
from google.colab import drive
drive.mount('/content/drive')

# Đổi "ten" thành tên bạn
!mkdir -p "/content/drive/MyDrive/phobert_ner_outputs"
!cp -r outputs/{ARCH} "/content/drive/MyDrive/phobert_ner_outputs/{ARCH}_ten"

Mounted at /content/drive


## 8. (Tuỳ chọn) Thử model với câu của bạn

In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
# !ls -lah /content/drive/MyDrive/phobert_ner_outputs/outputs/softmax_tuanloc_1

total 513M
drwx------ 2 root root 4.0K Nov  5 08:16 .
drwx------ 3 root root 4.0K Nov  4 15:59 ..
-rw------- 1 root root 513M Nov  4 15:59 best_model.pt
-rw------- 1 root root  73K Nov  4 15:59 confusion_matrix.png
-rw------- 1 root root  14K Nov  4 15:59 pred_test_hard.txt
-rw------- 1 root root  24K Nov  4 15:59 pred_test.txt
-rw------- 1 root root  979 Nov  4 15:59 report_test_hard.txt
-rw------- 1 root root  980 Nov  4 15:59 report_test.txt


In [18]:
CAU_CAN_THU = "tôi tên lộc, giỏi tiếng anh và đẹp trai"  #@param {type:"string"}

model_path = "/content/drive/MyDrive/phobert_ner_outputs/outputs/softmax_tuanloc_1/best_model.pt"

import os
import subprocess

# Kiểm tra model
if not os.path.exists(model_path):
    raise FileNotFoundError(f"Không tìm thấy model: {model_path}")

# Chạy dự đoán
cmd = f'printf "%s\\n" "{CAU_CAN_THU}" | python VPhoBertTagger/main.py predict --model_path "{model_path}"'

out = subprocess.run(
    cmd,
    shell=True,
    capture_output=True,
    text=True
)

# Hiển thị kết quả
for line in out.stdout.splitlines():
    if 'EOFError' not in line:
        print(line)

# Hiển thị lỗi nếu có
if out.returncode != 0:
    print("Lỗi khi chạy predict:")
    print(out.stderr)

[ViTagger] VnCoreNLP loading ...
[ViTagger] Model loading ...
[ViTagger] All ready!
Enter text:[('tiếng anh', 'SKILL')]
Enter text:
Lỗi khi chạy predict:
I0000 00:00:1793870872.687705   11383 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793870875.291286   11383 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Start PREDICT process...
INFO:vphoberttagger.helper:Start PREDICT process...
/content/training_model/phobert_ner/VPhoBertTagger/vphoberttagger/predictor.py:91: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:2

---
### Lỗi hay gặp
- `ModuleNotFoundError` bất kỳ module nào / `vncorenlp jar not found` / Colab tự ngắt kết nối giữa buổi
  → chạy lại cell 1, 2, 3 (runtime bị reset, mất hết file + package đã cài).
- `git push` báo lỗi quyền / authentication failed
  → bạn chưa được add làm collaborator trên GitHub repo, hoặc dùng mật khẩu thường thay vì Personal Access Token.